In [98]:
import nltk
import json
import pandas as pd
import numpy as np
import seaborn as sns

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display
import matplotlib.pyplot as plt

from storage.database import Database

In [4]:
BASE_URL = "https://dadosabertos.camara.leg.br/api/v2/"
DATABASE_PATH = "../data.db"


try:
    stopwords.words('portuguese')
except LookupError:
    nltk.download('stopwords')
nltk.download('rslp')

[nltk_data] Downloading package rslp to
[nltk_data]     C:\Users\micael.conti\AppData\Roaming\nltk_data...
[nltk_data]   Package rslp is already up-to-date!


True

In [68]:
database = Database(DATABASE_PATH)

In [77]:
def mostrar_tabela(x,vec,title):
    df_bow = pd.DataFrame(x.toarray(), columns=vec.get_feature_names_out())
    print(title)
    display(df_bow)

In [84]:
def processar_dados_bow(campo):
    discursos = database.get_discursos(10)
    
    corpus = [json.loads(d[campo]) for d in discursos]
    
    bow_vec = CountVectorizer(
        tokenizer=lambda x: x,
        preprocessor=None,
        lowercase=False
    )
    
    X_bow = bow_vec.fit_transform(corpus)
    
    return X_bow, bow_vec,corpus

In [85]:
X_bow,bow_vec,bow_corpus = processar_dados_bow("sumario_stemizado")
mostrar_tabela(X_bow,bow_vec,"Matriz Documento-Termo (BoW):")

Matriz Documento-Termo (BoW):


c:\GIT\scrap\Trabalho\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,abr,acess,aere,aeroport,afet,agenc,agradec,aguinald,alhei,aliquot,...,urgenc,uso,val,valorizaca,vet,viag,vid,viv,voo,votaca
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
1,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,1,0,...,0,0,1,0,0,0,1,1,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,3,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
5,1,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
7,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1
9,0,0,0,0,1,1,1,0,0,0,...,0,0,0,0,1,1,0,0,0,0


In [91]:
def processar_dados_td_idf(campo,size):
    discursos = database.get_discursos(size)
    
    corpus = [
        json.loads(d[campo])
        for d in discursos
        if campo in d and d[campo]
    ]    
    
    tfidf_vec = TfidfVectorizer(
        lowercase=False,
        preprocessor=None, # Usa a lista de stopwords corrigida
        tokenizer=lambda x: x
    )
    
    X_tfidf = tfidf_vec.fit_transform(corpus)
    
    return X_tfidf, tfidf_vec, corpus

In [ ]:
x1,vec1,corpus1 = processar_dados_td_idf("sumario_tokens",100)
x2,vec2,corpus2 = processar_dados_td_idf("sumario_stemizado",1000)
x3,vec3,corpus3 = processar_dados_td_idf("sumario_lemizado",1000)

mostrar_tabela(x1,vec1,"Matriz Documento-Termo (TF-IDF)")

Matriz Documento-Termo (TF-IDF)


,abaixo,abaixoassinados,abandonadas,abandonados,abandonar,abandono,abastecem,abastecimento,abc,abencoando,...,zema,zepbound,zerados,zerar,zeze,zfm,zion,zohan,zona,zumbi
0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
969,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
970,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
971,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
972,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [87]:
def search_and_rank(query, vectorizer, X_corpus, corpus, method_name):
    """
    Vetoriza uma query, calcula a similaridade com o corpus e exibe os resultados.
    """
    q_vec = vectorizer.transform([query])
    sim_scores = cosine_similarity(q_vec, X_corpus).ravel()
    rank = np.argsort(sim_scores)[::-1]

    print(f"Top-3 Similares para a Query (usando {method_name}):")
    for i in rank[:3]:
        if sim_scores[i] > 0.01: # Apenas mostra se houver alguma similaridade
            print(f"  Doc{i+1} (score={sim_scores[i]:.3f}): {corpus[i]}")
    print("-" * 40)

In [99]:
query = "deputado que gosta de dinheiro"

print(f">> Executando busca para a query: '{query}'\n")
search_and_rank(query, bow_vec, X_bow, bow_corpus, "BoW")
search_and_rank(query, vec1, x1, corpus1, "TF-IDF")

>> Executando busca para a query: 'deputado que gosta de dinheiro'

Top-3 Similares para a Query (usando BoW):
  Doc4 (score=0.260): ['deput', 'encaminh', 'votaca', 'recurs', 'apreciaca', 'termin', 'incis', 'i', 'caput', 'art', 'd', 'lei', 'n', 'cont', 'art', 'substitu', 'sen', 'feder', 'apresent', 'projet', 'lei', 'n', 'alt', 'lei', 'n', 'dezembr', 'lei', 'diretriz', 'bas', 'educaca', 'nacion', 'defin', 'diretriz', 'ensin', 'medi']
  Doc7 (score=0.091): ['deput', 'discut', 'projet', 'lei', 'n', 'alt', 'lei', 'n', 'mai', 'estabelec', 'aliquot', 'reduz', 'ambit', 'program', 'emerg', 'retom', 'set', 'event', 'pers']
  Doc9 (score=0.058): ['deput', 'encaminh', 'votaca', 'requer', 'urgenc', 'apreciaca', 'projet', 'lei', 'n', 'dispo', 'criaca', 'dia', 'rei', 'pel']
----------------------------------------
Top-3 Similares para a Query (usando TF-IDF):
  Doc4 (score=0.287): ['deputado', 'encaminhou', 'votacao', 'recurso', 'apreciacao', 'terminativa', 'inciso', 'i', 'caput', 'art', 'd', 'lei',

In [ ]:
print("Calculando a matriz de similaridade de cossenos...")
sim_matrix = cosine_similarity(x1, x1)

# Para melhor visualização, criamos um DataFrame com os rótulos corretos
doc_labels = [f"Doc{i+1}" for i in range(len(corpus1))]
df_similarity = pd.DataFrame(sim_matrix, index=doc_labels, columns=doc_labels)

# Agora, geramos o heatmap a partir dessa matriz de similaridade
plt.figure(figsize=(8, 6))
sns.heatmap(
    df_similarity,
    annot=True,          # Exibe os valores de similaridade nas células
    cmap="Blues",        # Mapa de cores em tons de azul
    fmt=".2f"            # Formata os números com duas casas decimais
)
plt.title("Heatmap de Similaridade de Cossenos entre Documentos")
plt.xlabel("Documentos")
plt.ylabel("Documentos")
plt.tight_layout()
plt.show()

Calculando a matriz de similaridade de cossenos...


KeyboardInterrupt: 